# 01 · Data inventory and quality checks (train + student_test)

In [1]:
import sys, os
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "pathsnap").is_dir()), None)
if _root is None:
    raise RuntimeError("src/pathsnap not found: unzip all zips of the set into the same pathsnap/ folder")
sys.path.insert(0, str(_root / "src")); os.chdir(_root / "notebooks")
import numpy as np, pandas as pd
from pathsnap import io

In [2]:
rows = []
for split in ["train", "student_test"]:
    for tid in io.list_trips(split):
        p = os.path.join(io.split_dir(split), tid, tid)
        track, nodes, arcs, route = io.load_raw(split, tid)
        t = track[:, 2]
        dt = np.diff(t)
        clean = io.clean_track(track)
        xy = io.project(clean[:, 0], clean[:, 1], clean[:, 0].mean(), clean[:, 1].mean())
        rows.append(dict(
            split=split, trip=tid,
            files=sum(os.path.exists(p + e) for e in [".track", ".nodes", ".arcs", ".route"]),
            track_kb=os.path.getsize(p + ".track") / 1024, nodes_kb=os.path.getsize(p + ".nodes") / 1024,
            arcs_kb=os.path.getsize(p + ".arcs") / 1024,
            n_obs=len(track), n_nan=int(np.isnan(track).sum()), n_dup_t=int(len(t) - len(np.unique(t))),
            sorted=bool((dt >= 0).all()), dt_1s=float(np.mean(dt == 1)), dt_max=float(dt.max()),
            n_gaps_10s=int((dt > 10).sum()), duration_s=float(t[-1] - t[0]),
            n_nodes=len(nodes), n_arcs=len(arcs), route_len=0 if route is None else len(route),
            lon_c=clean[:, 0].mean(), lat_c=clean[:, 1].mean(),
            extent_km=float(np.ptp(xy, 0).max() / 1000),
        ))
inv = pd.DataFrame(rows)
os.makedirs("../outputs", exist_ok=True)
inv.to_csv("../outputs/inventory.csv", index=False)
inv.head()

,split,trip,files,track_kb,nodes_kb,arcs_kb,n_obs,n_nan,n_dup_t,sorted,dt_1s,dt_max,n_gaps_10s,duration_s,n_nodes,n_arcs,route_len,lon_c,lat_c,extent_km
0,train,00000001,4,40.014648,1647.823242,1902.806641,1315,0,0,True,0.980974,16.0,1,1357.000,88809,165790,525,8.605599,50.088368,40.900775
1,train,00000002,4,232.504883,155.000000,152.850586,7243,0,0,True,0.974455,160.0,41,8933.156,7936,16139,262,11.032568,47.683925,4.127005
2,train,00000003,4,25.550781,178.476562,171.614258,821,0,0,True,0.996341,2.0,0,823.000,9138,18028,79,37.177362,55.987684,3.385890
3,train,00000004,4,34.162109,143.554688,146.228516,1064,0,0,True,0.538100,51.0,215,5217.000,7350,15499,320,12.424218,52.641499,14.305469
4,train,00000005,4,32.595703,17.539062,12.061523,1045,0,0,True,1.000000,1.0,0,1044.000,898,1579,105,35.503261,49.486068,28.410788


In [3]:
inv.groupby("split").agg(trips=("trip", "count"), obs=("n_obs", "sum"), files=("files", "min"),
                         nan=("n_nan", "sum"), dup_t=("n_dup_t", "sum"), unsorted=("sorted", lambda s: (~s).sum()))

,trips,obs,files,nan,dup_t,unsorted
split,,,,,,
student_test,10,19160,4,0,0,0
train,80,208417,4,0,0,0


In [4]:
inv[["n_obs", "duration_s", "dt_1s", "dt_max", "n_gaps_10s", "n_nodes", "n_arcs", "route_len", "extent_km"]].describe().round(2)

,n_obs,duration_s,dt_1s,dt_max,n_gaps_10s,n_nodes,n_arcs,route_len,extent_km
count,90.00,90.00,90.00,90.00,90.00,90.00,90.00,90.00,90.00
mean,2528.63,5274.25,0.94,2408.52,6.86,22490.82,45132.72,386.81,17.04
std,3157.00,17240.02,0.11,16996.22,24.97,30868.53,61920.28,359.24,18.25
min,262.00,265.00,0.53,1.00,0.00,299.00,649.00,59.00,1.13
25%,899.25,1133.80,0.96,1.00,0.00,3655.25,7282.75,166.75,4.35
50%,1496.50,1710.00,0.99,7.00,0.00,9305.00,18761.50,272.50,8.40
75%,2816.25,3618.75,1.00,67.25,5.00,26296.50,52913.25,464.25,22.92
max,19313.00,154030.00,1.00,153064.00,215.00,155003.00,307392.00,2266.00,75.40


In [5]:
print(inv[["track_kb", "nodes_kb", "arcs_kb"]].sum().round(0))
print(io.load_raw("train", inv.trip[0])[0].dtype, "-> columns: lon, lat, t | nodes: lon, lat | arcs: from, to | route: arc_id")

track_kb     7323.0
nodes_kb    39310.0
arcs_kb     45088.0
dtype: float64
float64 -> columns: lon, lat, t | nodes: lon, lat | arcs: from, to | route: arc_id


Trips with large GPS gaps (> 60 s)

In [6]:
inv[inv.dt_max > 60][["split", "trip", "n_obs", "dt_max", "n_gaps_10s", "duration_s"]].sort_values("dt_max", ascending=False)

,split,trip,n_obs,dt_max,n_gaps_10s,duration_s
72,train,00000090,536,153064.0,6,154030.000
61,train,00000078,2909,52810.0,25,57836.000
55,train,00000069,2067,2658.0,2,5558.000
16,train,00000021,15783,2161.0,2,18605.000
8,train,00000011,2828,909.0,1,3735.000
34,train,00000042,7739,670.0,37,12377.000
64,train,00000081,3224,536.0,86,8481.000
18,train,00000024,801,458.0,2,1541.000
36,train,00000044,546,341.0,2,980.000
67,train,00000084,16031,312.0,4,16524.000
